In [2]:
print("ok")

ok


In [3]:
%pwd

'c:\\Users\\Adama Camara\\immigration-of-canada\\research'

In [4]:
import os 
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\Adama Camara\\immigration-of-canada'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

c:\Users\Adama Camara\miniconda3\envs\immigbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [8]:
extracted_data=load_pdf_files("data")

In [9]:
extracted_data

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-02-12T22:25:27+00:00', 'author': 'Business Council of Alberta', 'keywords': 'DAFuWFR4lo0,BADf1_2MuTE', 'moddate': '2024-02-12T15:36:22-07:00', 'title': 'BCA_Prosperity Driven Immigration_National Strategy Ch1 - FINAL', 'source': 'data\\BCA_Prosperity-Driven-Immigration_National-Strategy-Ch1-FINAL.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='The Evolution ofImmigrationPolicy in Canada\nA Prosperity-Driven ImmigrationStrategy for Canada\nCanada’s National ImmigrationStrategy | Chapter One\nNovember 2023'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-02-12T22:25:27+00:00', 'author': 'Business Council of Alberta', 'keywords': 'DAFuWFR4lo0,BADf1_2MuTE', 'moddate': '2024-02-12T15:36:22-07:00', 'title': 'BCA_Prosperity Driven Immigration_National Strategy Ch1 - FINAL', 'source': 'data\\BCA_Prosperity-Driven-Immigration_National-Strategy-Ch1-FINAL.p

In [10]:
len(extracted_data)  

11

In [11]:
from typing import List
from langchain.schema import Document


def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [12]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [13]:
minimal_docs

[Document(metadata={'source': 'data\\BCA_Prosperity-Driven-Immigration_National-Strategy-Ch1-FINAL.pdf'}, page_content='The Evolution ofImmigrationPolicy in Canada\nA Prosperity-Driven ImmigrationStrategy for Canada\nCanada’s National ImmigrationStrategy | Chapter One\nNovember 2023'),
 Document(metadata={'source': 'data\\BCA_Prosperity-Driven-Immigration_National-Strategy-Ch1-FINAL.pdf'}, page_content='Copyright\nBusiness Council of Alberta 2023\nReport Citation\n“The Evolution of Immigration Policy in Canada. Canada’s National Immigration Strategy: Chapter One” \nBusiness Council of Alberta. November 2023.\nAbout the Business Council of Alberta\nThe Business Council of Alberta is a non-partisan, for-purpose organization dedicated to building a better \nAlberta within a more dynamic Canada. Composed of the chief executives and leading entrepreneurs of \nthe province’s largest enterprises, Council members are proud to represent the majority of Alberta’s \nprivate sector investment, job

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )

    texts = text_splitter.split_documents(minimal_docs)
    return texts

In [15]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 56


In [16]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\Adama Camara\AppData\Local\Temp\ipykernel_6456\30168171.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [17]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [18]:
vector = embedding.embed_query("Hello world")
vector

[-0.034477293491363525,
 0.031023185700178146,
 0.006734919268637896,
 0.026108955964446068,
 -0.03936200588941574,
 -0.16030244529247284,
 0.06692398339509964,
 -0.00644145580008626,
 -0.047450482845306396,
 0.014758873730897903,
 0.07087531685829163,
 0.05552761256694794,
 0.019193336367607117,
 -0.026251327246427536,
 -0.010109526105225086,
 -0.026940451934933662,
 0.02230745740234852,
 -0.02222668007016182,
 -0.14969263970851898,
 -0.017492998391389847,
 0.007676251698285341,
 0.05435226485133171,
 0.003254401497542858,
 0.031725890934467316,
 -0.08462139964103699,
 -0.029405971989035606,
 0.051595598459243774,
 0.04812406003475189,
 -0.003314854810014367,
 -0.05827920511364937,
 0.04196922481060028,
 0.022210687398910522,
 0.1281888335943222,
 -0.02233893983066082,
 -0.011656275019049644,
 0.06292839348316193,
 -0.032876357436180115,
 -0.0912260189652443,
 -0.03117534890770912,
 0.05269956961274147,
 0.04703487083315849,
 -0.08420306444168091,
 -0.030056191608309746,
 -0.020744830

In [19]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [20]:
print(os.getenv("PINECONE_API_KEY") is not None)
print(os.getenv("OPENAI_API_KEY") is not None)

True
True


In [21]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [22]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [28]:
from pinecone import ServerlessSpec

index_name = "immigration-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=11,  # Dimension of the embeddings
        metric="cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [29]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [30]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore

# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [31]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs = retriever.invoke("En quelle année le Canada a-t-il introduit son premier système de sélection des immigrants basé sur des points ?")
retrieved_docs

[Document(id='c9dee4c6-8b9a-4506-b51e-5d0991909cea', metadata={'source': 'data\\BCA_Prosperity-Driven-Immigration_National-Strategy-Ch1-FINAL.pdf'}, page_content='prospective immigrants, emphasizing their ability to \nbe successful economic contributors to Canada, \nconsidering factors such as education, occupational \nskills, employment prospects, age, and proficiency in \nofficial languages. The points assigned to specific \noccupations were revised regularly to address labour \nmarket needs. Notably, the point system did not take \ninto account ethnicity or place of birth, aiming to \nensure fairness and equity in the immigration \nprocess.'),
 Document(id='a9d1e109-fdae-460f-860d-503e70fd5e87', metadata={'source': 'data\\BCA_Prosperity-Driven-Immigration_National-Strategy-Ch1-FINAL.pdf'}, page_content='prospective immigrants, emphasizing their ability to \nbe successful economic contributors to Canada, \nconsidering factors such as education, occupational \nskills, employment prosp

In [33]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [45]:
from langchain.chains import create_retrieval_chain

from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

In [46]:
system_prompt = (
    "You are an immigration canada assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [47]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [48]:
response = rag_chain.invoke({"input": " Compare les changements apportés à Entrée express en 2016 et en 2023?"})
print(response["answer"])

Je suis désolé, mais je n'ai pas d'informations spécifiques sur les changements apportés à Entrée express en 2016 et en 2023. Je vous recommande de vérifier les annonces officielles du gouvernement canadien pour les détails les plus récents.


In [49]:
response = rag_chain.invoke({"input": " First Regulatory Structurefor Immigration"})
print(response["answer"])

The first regulatory structure for immigration in Canada was established between 1919 and 1929. During this period, immigration from preferred countries slowed, and immigrants from some non-preferred countries were admitted only if sponsored by certain relatives in Canada. The concept of "absorptive capacity" was also introduced to consider the economy’s ability to employ new immigrants at prevailing wages.
